In [11]:

import pandas as pd
import numpy as np
import joblib
import os


from sklearn.linear_model import LinearRegression

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)



# ==========================
# Load Processed Data
# ==========================


X_train = pd.read_csv(
    "../data/processed/X_train.csv"
)


X_test = pd.read_csv(
    "../data/processed/X_test.csv"
)


y_train = pd.read_csv(
    "../data/processed/y_train.csv"
).squeeze()


y_test = pd.read_csv(
    "../data/processed/y_test.csv"
).squeeze()



print("Original Shapes")
print("----------------")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)



# ==========================
# Target Cleaning
# ==========================


y_train = pd.to_numeric(
    y_train,
    errors="coerce"
)


y_test = pd.to_numeric(
    y_test,
    errors="coerce"
)



# Remove missing target rows

train_mask = y_train.notna()

test_mask = y_test.notna()



X_train = X_train.loc[
    train_mask
]


y_train = y_train.loc[
    train_mask
]



X_test = X_test.loc[
    test_mask
]


y_test = y_test.loc[
    test_mask
]



# ==========================
# Feature Cleaning
# ==========================


# Replace infinite values

X_train = X_train.replace(
    [np.inf,-np.inf],
    np.nan
)


X_test = X_test.replace(
    [np.inf,-np.inf],
    np.nan
)



# Fill missing values

X_train = X_train.fillna(0)

X_test = X_test.fillna(0)



# Align columns

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)



print("\nAfter Cleaning")
print("----------------")

print("X_train:",X_train.shape)

print("X_test :",X_test.shape)


print(
    "NaN in X_train:",
    X_train.isnull().sum().sum()
)


print(
    "NaN in X_test:",
    X_test.isnull().sum().sum()
)



# ==========================
# Baseline Models
# ==========================


models = {

    "Linear Regression":

    LinearRegression(),


    "Random Forest":

    RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),


    "Gradient Boosting":

    GradientBoostingRegressor(
        random_state=42
    )

}



baseline_results=[]



for name,model in models.items():


    print("\nTraining:",name)


    model.fit(
        X_train,
        y_train
    )


    prediction = model.predict(
        X_test
    )


    results={

        "Model":name,


        "MAE":

        mean_absolute_error(
            y_test,
            prediction
        ),


        "RMSE":

        np.sqrt(
            mean_squared_error(
                y_test,
                prediction
            )
        ),


        "R2 Score":

        r2_score(
            y_test,
            prediction
        )

    }


    baseline_results.append(results)




baseline_results=pd.DataFrame(
    baseline_results
)



print("\nBaseline Results")

display(
    baseline_results
)



baseline_results.to_csv(
    "../data/model_comparison_results.csv",
    index=False
)




# ==========================
# Hyperparameter Tuning
# Gradient Boosting
# ==========================



gb_model = GradientBoostingRegressor(
    random_state=42
)



param_grid = {


    "n_estimators":
    [
        100,
        200
    ],


    "learning_rate":
    [
        0.05,
        0.1
    ],


    "max_depth":
    [
        3,
        5
    ]

}



grid_search = GridSearchCV(

    estimator=gb_model,

    param_grid=param_grid,

    cv=3,

    scoring="neg_root_mean_squared_error",

    n_jobs=-1

)



print("\nStarting Grid Search...")



grid_search.fit(

    X_train,

    y_train

)



print("\nBest Parameters")

print(
    grid_search.best_params_
)



print("\nBest CV RMSE")

print(
    -grid_search.best_score_
)





# ==========================
# Final Model Evaluation
# ==========================



best_model = grid_search.best_estimator_



y_pred = best_model.predict(
    X_test
)



final_mae = mean_absolute_error(
    y_test,
    y_pred
)


final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)


final_r2 = r2_score(
    y_test,
    y_pred
)



print("\nFinal Tuned Gradient Boosting Results")

print("--------------------------------------")


print(
    "MAE:",
    final_mae
)


print(
    "RMSE:",
    final_rmse
)


print(
    "R2 Score:",
    final_r2
)




# ==========================
# Save Model
# ==========================


os.makedirs(
    "../models",
    exist_ok=True
)



joblib.dump(

    best_model,

    "../models/tuned_gradient_boosting_model.pkl"

)



print("\nModel Saved Successfully!")



# ==========================
# Save Predictions
# ==========================


prediction_results = pd.DataFrame({

    "Actual_Log_Price":

    y_test.values,


    "Predicted_Log_Price":

    y_pred

})



prediction_results.to_csv(

    "../data/predictions.csv",

    index=False

)



print("Prediction file saved!")

Original Shapes
----------------
X_train: (49792, 160)
X_test : (12448, 160)
y_train: (49792,)
y_test : (12448,)

After Cleaning
----------------
X_train: (49792, 160)
X_test : (12448, 160)
NaN in X_train: 0
NaN in X_test: 0

Training: Linear Regression

Training: Random Forest

Training: Gradient Boosting

Baseline Results


,Model,MAE,RMSE,R2 Score
0,Linear Regression,0.292680,0.396112,0.743247
1,Random Forest,0.210746,0.312820,0.839871
2,Gradient Boosting,0.251070,0.348087,0.801730



Starting Grid Search...

Best Parameters
{'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}

Best CV RMSE
0.2849808349902998

Final Tuned Gradient Boosting Results
--------------------------------------
MAE: 0.16996774153893343
RMSE: 0.26880639898804354
R2 Score: 0.8817610897325937

Model Saved Successfully!
Prediction file saved!
